In [2]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Script for scan [type of scan ...]

Initialize cax control of siriuspy devices

In [3]:
CAX = CAXCtrl()

# Functions

Useful functions for the scan

In [4]:
def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': CAX.dvf_A1.acquisition_time,
        'dvf2': {
            'z_pos': CAX.dvf_B1.z_pos
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos
        }
    }

# Scan

Initial state

In [5]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20250829-133208', '20250829')

In [6]:
parameters0 = current_parameters()

with open(f'initial_state_{formatted_time}.txt','w') as f:
    f.write(str(parameters0))

print(parameters0)

{'slit1': {'top': 16.86, 'bottom': 34.2, 'left': 43.49, 'right': 46.27}, 'dvf1': 0.1, 'dvf2': {'z_pos': None}, 'mirror': {'ry': 0.35169675, 'tx': -3.8500000000000014e-05, 'y1': -0.5017925, 'y2': 1.1943553125, 'y3': -0.2924078125}}


Parameters

In [7]:
STEP = 0.0001
SCALE = 1

Tests

In [264]:
CAX.dvf_A1.cmd_acquire_on()

True

In [255]:
CAX.mirror.ry_pos = 0.3391237

In [ ]:
ry0 = CAX.mirror.ry_pos
print(f'initial ry: {ry0}')

initial ry: 0.33579000000000003


In [ ]:
CAX.mirror.ry_pos = ry0 + STEP

In [8]:
def fwhm(data):
    threshold = 0.5*np.max(data)
    mask = data > threshold
    return np.sum(mask) * SCALE

def peak(data):
    return np.max(data)

def position(data):
    return np.argmax(data) * SCALE

def move(step=STEP):
    CAX.mirror.ry_pos += step

    time.sleep(5)

    img = CAX.dvf_B1.image

    projx = np.sum(img,axis=0)
    projy = np.sum(img,axis=1)

    x = position(projx)
    y = position(projy)

    fwhmx = fwhm(projx)
    fwhmy = fwhm(projy)

    peakx = peak(projx)
    peaky = peak(projy)

    return [np.round(CAX.mirror.ry_pos,8), x, fwhmx, peakx, y, fwhmy, peaky]

In [90]:
img = CAX.dvf_B1.image

projx = np.sum(img,axis=0)
projy = np.sum(img,axis=1)

x = position(projx)
y = position(projy)

fwhmx = fwhm(projx)
fwhmy = fwhm(projy)

peakx = peak(projx)
peaky = peak(projy)

header = '#    Ry          X        fwhmx    peakx      Y        fwhmy   peaky'
table  = [[np.round(CAX.mirror.ry_pos,8), x, fwhmx, peakx, y, fwhmy, peaky]]

In [ ]:
dRy = 0.000

table.append(move(step=dRy))

In [202]:

f = open('table_ry_2025-07-24.dat',mode='w')
f.write(header+'\n')

print(header)
for line in table:
    f.write(' '.join([str(ela) for ela in line]) + '\n')
    print(*line,sep='      ')

f.close()

#    Ry          X        fwhmx    peakx      Y        fwhmy   peaky
0.33626625      751      139      94451      844      84      147769
0.3363615      742      153      87717      844      85      148532
0.33645675      954      139      74628      838      85      116547
0.336552      779      162      76666      842      83      138998
0.33664725      769      143      88766      844      84      142704
0.3367425      798      138      85060      844      84      133406
0.33683775      818      139      82100      842      84      128824
0.336933      828      140      80065      843      85      125720
0.33702825      838      140      77774      842      85      122247
0.3371235      857      140      75112      844      86      118172
0.33721875      883      141      73167      842      85      115469
0.337695      1013      133      86536      842      84      132064
0.33817125      1259      132      135383      852      82      209216
0.3386475      1465      125      148084

In [15]:
amplitude_ry = 0.343029 - 0.3336945
amplitude_ry * 1e3

9.334499999999968

In [ ]:
img = CAX.dvf_B1.image

projx = np.sum(img,axis=0)
projy = np.sum(img,axis=1)

In [23]:
fwhmx = fwhm(projx)
fwhmy = fwhm(projy)

peakx = peak(projx)
peaky = peak(projy)

print(f'fwhm:\n x: {fwhmx} um\n y: {fwhmy} um')
print()
print(f'peak:\n x: {peakx}\n y: {peaky}')

fwhm:
 x: 66.72 um
 y: 41.28 um

peak:
 x: 96777
 y: 149720


Initialize scan file

In [55]:
filename = f'scan_ry_{formatted_date}_01.h5'
filedir = f"/home/ids/data/{formatted_date}-Slit-Mirror"
file = HDF5File(filename=filename,filedir=filedir)

file.save_metadata(metadata_dict={
    'ry': CAX.mirror.ry_pos
})

Execution

In [53]:
ry0 = CAX.mirror.ry_pos

In [18]:
steps1 = np.linspace(0, amplitude_ry, 101)[1:]
steps2 = steps1.copy()[::-1][1:]
steps = np.hstack([steps1,steps2])

In [249]:
steps*1e4

array([ 0.93345,  1.8669 ,  2.80035,  3.7338 ,  4.66725,  5.6007 ,
        6.53415,  7.4676 ,  8.40105,  9.3345 , 10.26795, 11.2014 ,
       12.13485, 13.0683 , 14.00175, 14.9352 , 15.86865, 16.8021 ,
       17.73555, 18.669  , 19.60245, 20.5359 , 21.46935, 22.4028 ,
       23.33625, 24.2697 , 25.20315, 26.1366 , 27.07005, 28.0035 ,
       28.93695, 29.8704 , 30.80385, 31.7373 , 32.67075, 33.6042 ,
       34.53765, 35.4711 , 36.40455, 37.338  , 38.27145, 39.2049 ,
       40.13835, 41.0718 , 42.00525, 42.9387 , 43.87215, 44.8056 ,
       45.73905, 46.6725 , 47.60595, 48.5394 , 49.47285, 50.4063 ,
       51.33975, 52.2732 , 53.20665, 54.1401 , 55.07355, 56.007  ,
       56.94045, 57.8739 , 58.80735, 59.7408 , 60.67425, 61.6077 ,
       62.54115, 63.4746 , 64.40805, 65.3415 , 66.27495, 67.2084 ,
       68.14185, 69.0753 , 70.00875, 70.9422 , 71.87565, 72.8091 ,
       73.74255, 74.676  , 75.60945, 76.5429 , 77.47635, 78.4098 ,
       79.34325, 80.2767 , 81.21015, 82.1436 , 83.07705, 84.01

In [251]:
( steps[1:] - steps[:-1] ) * 1e4

array([ 0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93345,
        0.93345,  0.93345,  0.93345,  0.93345,  0.93345,  0.93

In [252]:
len(steps)*5.8/60

19.236666666666668

In [254]:
0.2366*60

14.196

In [56]:
t0 = time.time()


for i, step in enumerate(steps1):
    print(f'{i}/{len(steps1)-1}')

    #!
    #todo: deslocar feixe ate a borda. como está agora é
    #todo: a partir do valor inicial, mas deve ser a partir da borda

    CAX.mirror.ry_pos = ry0 + step
    time.sleep(5)

    ry = CAX.mirror.ry_pos
    img1 = CAX.dvf_A1.image
    expotime1 = CAX.dvf_A1.exposure_time
    img2 = CAX.dvf_B1.image
    expotime2 = CAX.dvf_B1.exposure_time

    grpname = f'scan-{i:03d}'
    file.save_group(grpname=grpname, grpmetadata={'ry_pos':ry})
    file.save_dataset(grpname=grpname, dsetname='dvf1', 
                      dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
    file.save_dataset(grpname=grpname, dsetname='dvf2', 
                      dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print(f'elapsed time [s]: {t1-t0}')

0/99
1/99
2/99
3/99
4/99
5/99
6/99
7/99
8/99
9/99
10/99
11/99
12/99
13/99
14/99
15/99
16/99
17/99
18/99
19/99
20/99
21/99
22/99
23/99
24/99
25/99
26/99
27/99
28/99
29/99
30/99
31/99
32/99
33/99
34/99
35/99
36/99
37/99
38/99
39/99
40/99
41/99
42/99
43/99
44/99
45/99
46/99
47/99
48/99
49/99
50/99
51/99
52/99
53/99
54/99
55/99
56/99
57/99
58/99
59/99
60/99
61/99
62/99
63/99
64/99
65/99
66/99
67/99
68/99
69/99
70/99
71/99
72/99
73/99
74/99
75/99
76/99
77/99
78/99
79/99
80/99
81/99
82/99
83/99
84/99
85/99
86/99
87/99
88/99
89/99
90/99
91/99
92/99
93/99
94/99
95/99
96/99
97/99
98/99
99/99
elapsed time [s]: 546.779130935669


In [65]:
CAX.mirror.ry_pos

0.350649

In [ ]:
CAX.mirror.ry_pos += 5*STEP

In [66]:
CAX.mirror.ry_pos = 0.3512205